# 02 — Recursive Chunking

This notebook implements **Phase 03 only**. It reads the structured OpenStax page records produced in Phase 02, uses LangChain's `RecursiveCharacterTextSplitter` to compare practical chunk-size and overlap configurations, shows chunk examples, preserves provenance, and saves reusable chunk records.

> **Scope boundary:** This notebook does not create embeddings, Qdrant collections, retrieval, BM25, reranking, LLM calls, prompts, LangGraph graphs, or other RAG components.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from statistics import mean
from typing import Any

from IPython.display import JSON, Markdown, display
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_NAME = 'introduction_to_business_page_records.jsonl'
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'data' / 'processed' / INPUT_NAME).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f'Could not find data/processed/{INPUT_NAME} from {Path.cwd()}')

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / INPUT_NAME
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
FINAL_CHUNKS_PATH = OUTPUT_DIR / 'introduction_to_business_chunks.jsonl'
SUMMARY_PATH = OUTPUT_DIR / 'introduction_to_business_chunking_summary.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Phase 02 input: {INPUT_PATH}')

Project root: /home/ubuntu/business-knowledge-ai
Phase 02 input: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_page_records.jsonl


## 1. Load the Phase 02 page records

Only records with non-empty `cleaned_text` are candidates for chunking. Empty pages remain accounted for in the source artifact and are deliberately not turned into empty chunks.

In [2]:
with INPUT_PATH.open(encoding='utf-8') as handle:
    page_records = [json.loads(line) for line in handle if line.strip()]

required_page_fields = {'page_number', 'chapter', 'section', 'source', 'cleaned_text', 'quality_flags'}
assert page_records, 'Phase 02 produced no page records.'
assert all(required_page_fields <= record.keys() for record in page_records)

chunkable_pages = [record for record in page_records if record['cleaned_text'].strip()]
display(JSON({
    'phase_02_page_record_count': len(page_records),
    'pages_with_cleaned_text': len(chunkable_pages),
    'empty_pages_skipped': [record['page_number'] for record in page_records if not record['cleaned_text'].strip()],
    'inherited_fields': sorted(required_page_fields),
}))

<IPython.core.display.JSON object>

## 2. Compare reasonable recursive-splitting configurations

The comparison varies both `chunk_size` and `chunk_overlap`. Splitting is page-bounded so each chunk inherits a single, auditable PDF page, chapter, section, and source record. The separator order prioritizes paragraphs, then lines, sentence boundaries, spaces, and finally character-level fallback.

In [3]:
CONFIGURATIONS = [
    {'name': 'compact', 'chunk_size': 500, 'chunk_overlap': 75},
    {'name': 'balanced', 'chunk_size': 900, 'chunk_overlap': 150},
    {'name': 'broad_context', 'chunk_size': 1400, 'chunk_overlap': 200},
]
SEPARATORS = ['\n\n', '\n', '. ', ' ', '']

def split_page(record: dict[str, Any], config: dict[str, Any]) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config['chunk_size'],
        chunk_overlap=config['chunk_overlap'],
        length_function=len,
        separators=SEPARATORS,
        is_separator_regex=False,
    )
    return [chunk.strip() for chunk in splitter.split_text(record['cleaned_text']) if chunk.strip()]

def make_chunks(config: dict[str, Any]) -> list[dict[str, Any]]:
    chunks: list[dict[str, Any]] = []
    for page_record in chunkable_pages:
        for ordinal, text in enumerate(split_page(page_record, config), start=1):
            chunks.append({
                'chunk_id': f"openstax-introduction-business-p{page_record['page_number']:04d}-c{ordinal:03d}",
                'text': text,
                'source': page_record['source'],
                'page': page_record['page_number'],
                'chapter': page_record['chapter'],
                'section': page_record['section'],
                'chunking_config': {'name': config['name'], 'chunk_size': config['chunk_size'], 'chunk_overlap': config['chunk_overlap']},
                'character_count': len(text),
            })
    return chunks

In [4]:
comparison_chunks = {config['name']: make_chunks(config) for config in CONFIGURATIONS}
comparison_rows = []
for config in CONFIGURATIONS:
    chunks = comparison_chunks[config['name']]
    lengths = [chunk['character_count'] for chunk in chunks]
    comparison_rows.append({
        'configuration': config['name'],
        'chunk_size': config['chunk_size'],
        'chunk_overlap': config['chunk_overlap'],
        'chunk_count': len(chunks),
        'mean_characters': round(mean(lengths), 1),
        'min_characters': min(lengths),
        'max_characters': max(lengths),
    })
display(JSON(comparison_rows))
assert all(row['chunk_count'] > 0 for row in comparison_rows)

<IPython.core.display.JSON object>

### Representative chunk examples

The examples below are taken from the first content-bearing page. The full artifact retains every chunk without truncation.

In [5]:
for config in CONFIGURATIONS:
    example = comparison_chunks[config['name']][0]
    display(Markdown(f"#### {config['name']} — {config['chunk_size']} characters / {config['chunk_overlap']} overlap"))
    display(JSON({
        'chunk_id': example['chunk_id'],
        'page': example['page'],
        'chapter': example['chapter'],
        'section': example['section'],
        'character_count': example['character_count'],
        'text_preview': example['text'][:750],
    }))

#### compact — 500 characters / 75 overlap

<IPython.core.display.JSON object>

#### balanced — 900 characters / 150 overlap

<IPython.core.display.JSON object>

#### broad_context — 1400 characters / 200 overlap

<IPython.core.display.JSON object>

## 3. Select a reusable final configuration

`balanced` (900 characters with 150-character overlap) is selected as a preliminary engineering default: it is materially less fragmented than the compact setting while retaining more local context than the broad setting. This choice is documented, deterministic, and intentionally subject to future evaluation; it is not based on retrieval or generation metrics.

In [6]:
FINAL_CONFIG = next(config for config in CONFIGURATIONS if config['name'] == 'balanced')
final_chunks = comparison_chunks[FINAL_CONFIG['name']]
required_chunk_fields = {'chunk_id', 'text', 'source', 'page', 'chapter', 'section'}
assert all(required_chunk_fields <= chunk.keys() for chunk in final_chunks)
assert len({chunk['chunk_id'] for chunk in final_chunks}) == len(final_chunks), 'Chunk IDs must be unique.'
assert all(chunk['text'].strip() for chunk in final_chunks)

display(JSON({
    'selected_configuration': FINAL_CONFIG,
    'final_chunk_count': len(final_chunks),
    'required_chunk_fields': sorted(required_chunk_fields),
    'selection_note': 'Preliminary context/fragmentation trade-off only; no retrieval or generation evaluation has occurred.',
}))

<IPython.core.display.JSON object>

## 4. Save the reusable chunk artifact and validate it

The JSONL artifact contains one provenance-preserving final chunk per line. The companion summary makes the configuration comparison and Phase 03 boundary inspectable without re-running the notebook.

In [7]:
with FINAL_CHUNKS_PATH.open('w', encoding='utf-8') as handle:
    for chunk in final_chunks:
        handle.write(json.dumps(chunk, ensure_ascii=False) + '\n')

summary = {
    'input_artifact': str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    'final_chunk_artifact': str(FINAL_CHUNKS_PATH.relative_to(PROJECT_ROOT)),
    'input_page_record_count': len(page_records),
    'chunked_page_count': len(chunkable_pages),
    'selected_configuration': FINAL_CONFIG,
    'configuration_comparison': comparison_rows,
    'final_chunk_count': len(final_chunks),
    'required_chunk_fields': sorted(required_chunk_fields),
    'phase_scope': 'Recursive chunking only; no embeddings, Qdrant, retrieval, BM25, reranking, LLM, or LangGraph.',
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

with FINAL_CHUNKS_PATH.open(encoding='utf-8') as handle:
    persisted_chunks = [json.loads(line) for line in handle if line.strip()]
assert len(persisted_chunks) == len(final_chunks)
assert all(required_chunk_fields <= chunk.keys() for chunk in persisted_chunks)
assert len({chunk['chunk_id'] for chunk in persisted_chunks}) == len(persisted_chunks)

display(JSON({
    'final_chunks_path': str(FINAL_CHUNKS_PATH.relative_to(PROJECT_ROOT)),
    'summary_path': str(SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    'final_chunk_count': len(persisted_chunks),
    'validation': 'passed',
}))

<IPython.core.display.JSON object>

## Phase 03 result

The final artifact contains reusable, deterministic chunks with `chunk_id`, `text`, `source`, `page`, `chapter`, and `section`. No downstream RAG component has been implemented.